# Segementation Model Finetuning


- Finetuned on Kaggle T4 GPU * 2
- Base Model DeepLabV3
- Training Dataset: If you need training dataset, please email to agarwalrachit399@gmail.com


In [ ]:
!pip install torch torchvision pycocotools matplotlib
!pip install opencv-python
!pip install roboflow

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50
import torchvision.transforms.functional as F
from pycocotools.coco import COCO
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, ann_path, transform=None, target_size=(512, 512)):
        self.img_dir = img_dir
        self.coco = COCO(ann_path)
        self.img_ids = list(self.coco.imgs.keys())
        self.transform = transform
        self.target_size = target_size

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        img_info = self.coco.loadImgs(img_id)[0]

        img_path = os.path.join(self.img_dir, img_info['file_name'])
        image = Image.open(img_path).convert("RGB")

        mask = np.zeros((img_info["height"], img_info["width"]), dtype=np.uint8)
        for ann in anns:
            if ann['iscrowd'] == 0:
                rle = self.coco.annToMask(ann)
                mask = np.maximum(mask, rle * ann['category_id'])

        image = image.resize(self.target_size)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

        image = F.to_tensor(image)
        mask = torch.as_tensor(mask, dtype=torch.long)

        return image, mask


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()


roboflow_api_key = os.getenv("ROBOFLOW_API_KEY")
workspace_id = os.getenv("WORKSPACE_ID")
project_id = os.getenv("PROJECT_ID")

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=roboflow_api_key)  
project = rf.workspace(workspace_id).project(project_id)  
dataset = project.version("3").download("coco-segmentation",location="roboflow_dataset")

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
train_dataset = COCOSegmentationDataset(
    img_dir="roboflow_dataset/train/",
    ann_path="roboflow_dataset/train/_annotations.coco.json"
)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

loading annotations into memory...
Done (t=0.13s)
creating index...
index created!


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Modfify to use MPS if running on Mac Silicon

model = deeplabv3_resnet50(pretrained=True)
model.classifier[4] = nn.Conv2d(256, 2, kernel_size=1) 
model.to(device)

# Fix for BatchNorm issues with small batch sizes
def set_bn_eval(m):
    if isinstance(m, nn.BatchNorm2d):
        m.eval()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 5 # Adjust as needed
model.train()
model.apply(set_bn_eval)  # Important: disables BatchNorm tracking

for epoch in range(epochs):
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for images, masks in loop:

        images, masks = images.to(device), masks.to(device)

        outputs = model(images)["out"]
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=loss.item())

torch.save(model.state_dict(), "speech_bubble_segmentor_coco.pth")
print("✅ Model saved as speech_bubble_segmentor_coco.pth")